Step 0: import libraries

In [3]:
import shutil, zipfile
import glob
import os, re
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

Step 1: Mount Google Drive

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Step 2: Copy the zip from Drive to local Colab disk (local I/O is faster)

In [6]:
DRIVE_ZIP_PATH = '/content/drive/MyDrive/tennis_data.zip'
LOCAL_ZIP_PATH = '/content/tennis_data.zip'

shutil.copy(DRIVE_ZIP_PATH, LOCAL_ZIP_PATH)

'/content/tennis_data.zip'

Step 3: Extract the outer zip (contains 60 daily zip files at its root)

In [7]:
with zipfile.ZipFile(LOCAL_ZIP_PATH, 'r') as z:
    z.extractall('/content/tennis_raw/')

In [8]:
day_zips = sorted(glob.glob('/content/tennis_raw/*.zip'))
print(len(day_zips))       # should be 60
print(day_zips[:5])

60
['/content/tennis_raw/20240201.zip', '/content/tennis_raw/20240202.zip', '/content/tennis_raw/20240203.zip', '/content/tennis_raw/20240204.zip', '/content/tennis_raw/20240205.zip']


Step 4: Function to read one "table type" (by filename prefix) for one day

raw_match_parquet mixes 10 different tables together in the same folder,
distinguished only by a filename prefix (e.g."event_12021435.parquet",
"home_team_score_12021435.parquet"). We use a strict regex (not a plain startswith) so that e.g. prefix "home_team" never matches files thatactually belong to "home_team_score".

In [9]:
def read_prefix_day(day_root, prefix, subfolder='raw_match_parquet'):
    pattern = re.compile(rf'^{re.escape(prefix)}_\d+\.parquet$')

    folder = os.path.join(day_root, 'data/raw', subfolder)
    matched = [os.path.join(folder, f) for f in os.listdir(folder) if pattern.match(f)]

    if not matched:
        return pd.DataFrame()

    # Read every small parquet file and merge them into one table.
    # promote_options='permissive' handles cases where the same column
    # has a slightly different type across files (e.g. height as int64
    # in some files and double in others) by promoting to a common type.
    tables = [pq.read_table(f) for f in matched]
    return pa.concat_tables(tables, promote_options='permissive').to_pandas()

Step 5: Extract all 60 daily zips, each into its own subfolder
(total extracted size is only ~4-5 GB, so we keep every day on disk at once instead of extracting/processing/deleting one at a time)

In [10]:
EXTRACT_ROOT = '/content/tennis_all_days/'

for zip_path in day_zips:
    day_name = os.path.splitext(os.path.basename(zip_path))[0]   # e.g. "20240201"
    dest = os.path.join(EXTRACT_ROOT, day_name)

    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(dest)

print("Done extracting all daily zips")

day_dirs = sorted(glob.glob('/content/tennis_all_days/*'))
print(len(day_dirs))       # should be 60
print(day_dirs[:3])

Done extracting all daily zips
60
['/content/tennis_all_days/20240201', '/content/tennis_all_days/20240202', '/content/tennis_all_days/20240203']


Step 6: Aggregate one table prefix across all 60 days

In [11]:
def read_prefix_all_days(day_dirs, prefix, subfolder='raw_match_parquet'):
    parts = []
    for day_root in day_dirs:
        df = read_prefix_day(day_root, prefix, subfolder)
        if not df.empty:
            parts.append(df)
    if not parts:
        return pd.DataFrame()
    return pd.concat(parts, ignore_index=True)

match_prefixes = ['event', 'home_team', 'away_team', 'home_team_score',
                   'away_team_score', 'tournament', 'season', 'round', 'venue', 'time']

match_tables = {}
for p in match_prefixes:
    match_tables[p] = read_prefix_all_days(day_dirs, p)
    print(p, '->', match_tables[p].shape)

event -> (35053, 10)
home_team -> (25610, 18)
away_team -> (24203, 18)
home_team_score -> (35164, 14)
away_team_score -> (35053, 14)
tournament -> (35671, 16)
season -> (35671, 4)
round -> (19283, 5)
venue -> (35423, 5)
time -> (35671, 7)


Step 7: Aggregate the 5 "flat" folders across all 60 days
(each folder holds a single table type -> one parquet file per match,
no prefix-splitting needed, but we still use the manual read+concat
approach with promote_options='permissive' because pyarrow.dataset
fails on boolean columns that are entirely null in some files, e.g.
'suspended' in raw_odds_parquet)

In [12]:
flat_folders = {
    'votes': 'raw_votes_parquet',
    'power': 'raw_tennis_power_parquet',
    'statistics': 'raw_statistics_parquet',
    'point_by_point': 'raw_point_by_point_parquet',
    'odds': 'raw_odds_parquet',
}

def read_flat_folder_all_days(day_dirs, subfolder):
    day_parts = []
    for day_root in day_dirs:
        folder = os.path.join(day_root, 'data/raw', subfolder)
        if not os.path.isdir(folder) or not os.listdir(folder):
            continue
        files = [os.path.join(folder, f) for f in os.listdir(folder) if f.endswith('.parquet')]
        tables = [pq.read_table(f) for f in files]
        day_table = pa.concat_tables(tables, promote_options='permissive')
        day_parts.append(day_table)

    if not day_parts:
        return pd.DataFrame()

    full_table = pa.concat_tables(day_parts, promote_options='permissive')
    return full_table.to_pandas()

flat_tables = {}
for name, folder in flat_folders.items():
    flat_tables[name] = read_flat_folder_all_days(day_dirs, folder)
    print(name, '->', flat_tables[name].shape)

votes -> (35658, 3)
power -> (469677, 5)
statistics -> (1358234, 13)
point_by_point -> (2549369, 13)
odds -> (60946, 11)



Step 8: Check for duplicate match_id values

A match can appear in more than one daily zip (e.g. scraped again on a
neighboring day). We verified earlier that duplicated match_id rows are
exact copies of each other (not partial/updated versions), so a plain
drop_duplicates on match_id is safe here.

In [13]:
print(match_tables['event']['match_id'].duplicated().sum())
print(match_tables['tournament']['match_id'].duplicated().sum())

18180
18798


Step 9: Deduplicate

In [14]:
for p in match_prefixes:
    before = match_tables[p].shape[0]
    match_tables[p] = match_tables[p].drop_duplicates(subset='match_id', keep='first')
    after = match_tables[p].shape[0]
    print(p, ':', before, '->', after)

# flat tables: match_id is not a unique key here (one match has many rows),
# so we compare full rows instead
for name in flat_tables:
    before = flat_tables[name].shape[0]
    flat_tables[name] = flat_tables[name].drop_duplicates(keep='first')
    after = flat_tables[name].shape[0]
    print(name, ':', before, '->', after)

event : 35053 -> 16873
home_team : 25610 -> 12389
away_team : 24203 -> 11690
home_team_score : 35164 -> 16873
away_team_score : 35053 -> 16873
tournament : 35671 -> 16873
season : 35671 -> 16873
round : 19283 -> 9243
venue : 35423 -> 16749
time : 35671 -> 16873
votes : 35658 -> 20715
power : 469677 -> 249587
statistics : 1358234 -> 746361
point_by_point : 2549369 -> 1254744
odds : 60946 -> 34807


Step 10: Build the unified master table
Start from 'event' (one row per match) and left-merge every other table on match_id. Columns other than match_id are prefixed with the table name first (e.g. height -> home_team_height / away_team_height) so columns from different tables never collide. how='left' keeps every match even if it's missing rows in some table (e.g. ~30-40% of matches have no home_team/away_team info).

In [15]:
master = match_tables['event'].copy()

other_prefixes = ['home_team', 'away_team', 'home_team_score', 'away_team_score',
                   'tournament', 'season', 'round', 'venue', 'time']

for p in other_prefixes:
    df = match_tables[p].copy()
    rename_map = {c: f'{p}_{c}' for c in df.columns if c != 'match_id'}
    df = df.rename(columns=rename_map)
    master = master.merge(df, on='match_id', how='left')

print(master.shape)          # (16873, 102)
print(master.columns.tolist())

(16873, 102)
['match_id', 'first_to_serve', 'home_team_seed', 'away_team_seed', 'custom_id', 'winner_code', 'default_period_count', 'start_datetime', 'match_slug', 'final_result_only', 'home_team_name', 'home_team_slug', 'home_team_gender', 'home_team_user_count', 'home_team_residence', 'home_team_birthplace', 'home_team_height', 'home_team_weight', 'home_team_plays', 'home_team_turned_pro', 'home_team_current_prize', 'home_team_total_prize', 'home_team_player_id', 'home_team_current_rank', 'home_team_name_code', 'home_team_country', 'home_team_full_name', 'away_team_name', 'away_team_slug', 'away_team_gender', 'away_team_user_count', 'away_team_residence', 'away_team_birthplace', 'away_team_height', 'away_team_weight', 'away_team_plays', 'away_team_turned_pro', 'away_team_current_prize', 'away_team_total_prize', 'away_team_player_id', 'away_team_current_rank', 'away_team_name_code', 'away_team_country', 'away_team_full_name', 'home_team_score_current_score', 'home_team_score_display_s